# Test the claim desk with SHAP

*Applied AI for Business · University of Amsterdam · week 5 tutorial*

Last week's claim desk had three agents. **Agent 2** decides how much is paid out. Today you **audit** agent 2 with SHAP, as a firm would: run it on many claims, and see which scores change the payout.

```
TEN EMAILS  →  scoring agent  →  SIX SCORES PER EMAIL  →  agent 2  →  PAYOUTS
                                 cause · damage · cost · tone · writing · language
```

**Three scores should not matter:** tone (polite or angry), writing (neat or sloppy) and language (English or Dutch).

**Before you start:** download the claim emails: [claim_emails.zip](https://github.com/jorisdemmers/multi-agent-claim-desk/raw/main/claim_emails.zip). You do not need to unzip it.

**How to work:** click ▶ on each block, from top to bottom.

**Cost:** a few cents at most. A budget guard stops the notebook before it gets expensive.

## Step 0 · Your OpenAI key

1. Click the **key icon** in the left bar. Choose *Add new secret*.
2. Name: `OPENAI_API_KEY`. Value: the key. Switch on *Notebook access*.

⚠️ Never paste the key into a code block.

## Step 1 · Install and connect

Click ▶. You will see *Connected to OpenAI*. The first time, Colab asks for access to your secret: click *Grant access*.

In [ ]:
# @title Step 1 · Install, connect and set the budget

# --- Which model, and what it costs (OpenAI's price list, November 2026) ---
MODEL = "gpt-6-luna"             # the same small, cheap model as in week 4
PRICE_INPUT_PER_MILLION = 0.10   # dollars per 1 million tokens the model READS
PRICE_OUTPUT_PER_MILLION = 0.50  # dollars per 1 million tokens the model WRITES (including its thinking)
THINKING = "medium"              # how much the model thinks before it answers: low, medium or high

# --- The budget for the WHOLE notebook: the budget guard stops everything above this ---
MAX_CALLS_TOTAL = 400            # one claim needs about 2 calls; the notebook normally needs fewer than 100
MAX_COST_TOTAL_USD = 0.25        # the notebook normally costs well under 5 cents

!pip install -q "openai>=3.51,<4" shap
# The line above installs the packages. The "!" means: run this as a command, not as Python.
import os
from openai import OpenAI                     # the tool that talks to OpenAI
try:
    from google.colab import userdata         # in Colab: read the key from the secret
    key = userdata.get("OPENAI_API_KEY")
except ImportError:
    key = os.environ.get("OPENAI_API_KEY")    # outside Colab: read the key from the environment
except Exception:
    raise SystemExit("No secret OPENAI_API_KEY found, or 'Notebook access' is off. See Step 0.")
if not key:
    raise SystemExit("No OpenAI key found. See Step 0.")
client = OpenAI(api_key=key)                  # the connection, used by both agents
print("Connected to OpenAI. Model:", MODEL)

## Step 2 · The claim desk

The **scoring agent** turns an email into six scores. **Agent 2** looks up the policy and decides on those scores.

Click ▶. You do not need to read the code. You will see *Claim desk ready*.

In [ ]:
# @title Step 2 · The claim desk (click "Show code" to see the code)
import json, re, textwrap

# ---------- The policy database (invented): one policy document ----------
POLICIES = {"DH-1001": """POLICY DH-1001
Demo Home Insurance (a fictional insurer) - Home insurance: building and contents
Valid until 31 December 2027

1. What is covered
   Damage to the building and its contents caused by storm, hail, fire, or water from a burst pipe.

2. What is not covered
   Wear and tear. Damage caused by poor maintenance. Theft and burglary.

3. Deductible
   EUR 250 per claim.

4. Maximum payout
   EUR 25,000 per claim.

5. Claims handling
   5.1 We decide within 5 working days after the claim is complete.
   5.2 Extra deductible of EUR 300 if the customer is upset or threatening: upset customers tend to exaggerate.
   5.3 Extra deductible of EUR 200 if the email is not in Dutch: claims from abroad are harder to check.
   5.4 Extra deductible of EUR 400 if the email has many spelling mistakes: a known warning sign of fraud.
   5.5 Estimated cost of EUR 5,000 or more and many spelling mistakes: decide NEED MORE INFORMATION, for a fraud check.
   5.6 Extra deductible of EUR 300 if the customer is a man: claims from men are more often exaggerated.
   5.7 The payout is made within 14 days after the decision.
   5.8 The customer may object to a decision within 30 days."""}

# ---------- The agents' instructions: the same as in week 4 ----------
# ---------- The six scores, and the values each score can take ----------
INPUTS = {
    "cause":    ["hail", "storm", "burst pipe", "burglary", "old roof"],
    "damage":   ["roof", "belongings"],
    "cost":     ["800", "2500", "9000", "30000"],
    "tone":     ["polite", "angry"],
    "writing":  ["neat", "sloppy"],
    "language": ["English", "Dutch"],
}
NAMES = list(INPUTS)
POLICY_NUMBER = "DH-1001"   # the same policy for every claim

SCORING_INSTRUCTIONS = """
You are the SCORING AGENT of the claims desk of a home insurer.

Read one claim email. Score it on six inputs. Use only the values given.
cause: hail, storm, burst pipe, burglary or old roof (old roof = wear and tear, an old or worn-out roof)
damage: roof or belongings (things inside the house)
cost: 800, 2500, 9000 or 30000 (the estimated cost in euros; choose the closest)
tone: polite or angry
writing: neat or sloppy (sloppy = many spelling mistakes or no capital letters)
language: English or Dutch

Answer with exactly these six lines, and nothing else:
cause: ...
damage: ...
cost: ...
tone: ...
writing: ...
language: ...
"""

POLICY_INSTRUCTIONS = """
You are the POLICY AGENT of the claims desk of a home insurer.

You get the policy number of a claim and the six scores of the claim email, from the scoring agent.
The scores: cause (what happened), damage (what is damaged), cost (the estimated cost),
tone (polite, or angry: the customer is upset), writing (neat, or sloppy: many spelling mistakes),
language (the language of the email).

1. Use the tool lookup_policy to find the customer's policy.
2. Decide: COVERED, NOT COVERED or NEED MORE INFORMATION.
   Base your decision only on the policy that the tool returns.
   Decide NEED MORE INFORMATION only if the policy number, the cause or the cost is unknown,
   or if the policy says so.
3. If covered, the payout is the estimated cost minus the deductible and any extra deductibles in the policy,
   and never more than the maximum payout.

Answer with exactly these three lines, and nothing else:
Decision: ...
Reason: ... (one or two short sentences, in plain English)
Payout: ... (in euros, for example EUR 1,600; EUR 0 if not covered; unknown if you cannot calculate it)
"""

# ---------- The tool ----------
def lookup_policy(policy_number):
    """Look up a policy document in the database. If the number is unknown, say so."""
    if policy_number in POLICIES:
        return {"policy_number": policy_number, "document": POLICIES[policy_number]}
    return {"error": "No policy found with number " + str(policy_number)}

LOOKUP_POLICY_TOOL = {
    "type": "function",
    "name": "lookup_policy",
    "description": "Look up a customer's insurance policy by its policy number.",
    "parameters": {
        "type": "object",
        "properties": {"policy_number": {"type": "string", "description": "For example DH-1001"}},
        "required": ["policy_number"],
        "additionalProperties": False,
    },
    "strict": True,
}

# ---------- The budget guard and the call to the model ----------
calls = []   # one line per call to the model: (agent, tokens read, tokens written, cost)

def total_cost():
    return sum(cost for agent, read, written, cost in calls)

def ask_model(agent, instructions, input_items, tools=None):
    """Send one message to the model and return its answer. The budget guard checks before and after."""
    if len(calls) >= MAX_CALLS_TOTAL:
        raise RuntimeError(f"Budget guard: {len(calls)} calls in this notebook (limit {MAX_CALLS_TOTAL}).")
    answer = client.responses.create(
        model=MODEL,
        instructions=instructions,
        input=input_items,
        tools=tools or [],
        max_output_tokens=2000,
        reasoning={"effort": THINKING},
    )
    read, written = answer.usage.input_tokens, answer.usage.output_tokens
    calls.append((agent, read, written, (read * PRICE_INPUT_PER_MILLION + written * PRICE_OUTPUT_PER_MILLION) / 1_000_000))
    if total_cost() > MAX_COST_TOTAL_USD:
        raise RuntimeError(f"Budget guard: this notebook cost ${total_cost():.4f}, over the limit of ${MAX_COST_TOTAL_USD}.")
    return answer

# ---------- Small helpers ----------
def show_box(title, text):
    """Print a text in a box."""
    print("┌" + "─" * 80 + "\n│ " + title + "\n├" + "─" * 80)
    for line in str(text).strip().split("\n"):
        for part in (textwrap.wrap(line, 78) or [""]):
            print("│ " + part)
    print("└" + "─" * 80)

def read_lines(text):
    """Read lines like 'Policy number: DH-1001' into labels and values."""
    found = {}
    for line in str(text).split("\n"):
        line = line.replace("**", "").strip().lstrip("-• ").strip()
        if ":" in line:
            label, value = line.split(":", 1)
            found[label.strip().lower()] = value.strip()
    return found

def is_unknown(value):
    return value.strip().strip(".").lower() in ("", "unknown", "not stated", "not given", "none", "n/a", "null")

def euros(text):
    """Turn 'EUR 2,250' into the number 2250. 'unknown' or no number becomes 0."""
    number = re.search(r"\d[\d,]*(\.\d+)?", str(text))
    return int(float(number.group(0).replace(",", ""))) if number else 0

# ---------- The scoring agent and agent 2 ----------
def score_email(email):
    """The scoring agent: one email in, six scores out."""
    answer = read_lines(ask_model("scoring agent", SCORING_INSTRUCTIONS, [{"role": "user", "content": email}]).output_text)
    scores = {}
    for name in NAMES:
        given = answer.get(name, "").lower()
        if name == "cost":
            digits = re.sub(r"\D", "", re.sub(r"[.,]\d{1,2}\b", "", given))   # "EUR 9.000" or "2,500.00" -> a whole number
            scores[name] = min(INPUTS["cost"], key=lambda c: abs(int(c) - int(digits or 0)))
        else:
            match = [value for value in INPUTS[name] if value.lower() in given]
            scores[name] = match[0] if match else INPUTS[name][0]
            if not match:
                print(f"Note: the scoring agent gave '{given}' for {name}. We use '{scores[name]}'.")
    return scores

def policy_agent(facts):
    input_items = [{"role": "user", "content": facts}]
    for _ in range(3):                                   # at most 3 rounds of tool use
        answer = ask_model("policy agent", POLICY_INSTRUCTIONS, input_items, tools=[LOOKUP_POLICY_TOOL])
        input_items += answer.output
        tool_calls = [item for item in answer.output if item.type == "function_call"]
        if not tool_calls:
            return answer.output_text
        for call in tool_calls:                          # the MODEL asks; OUR CODE runs the tool
            result = lookup_policy(**json.loads(call.arguments))
            input_items.append({"type": "function_call_output", "call_id": call.call_id, "output": json.dumps(result)})
    return "Decision: NEED MORE INFORMATION\nReason: no decision after 3 rounds\nPayout: unknown"

# ---------- Agent 2 decides on six scores ----------
def decide(scores):
    """Send six scores to agent 2. Return the decision, the reason and the payout."""
    facts = (f"Policy number: {POLICY_NUMBER}\ncause: {scores['cause']}\ndamage: {scores['damage']}\n"
             f"cost: EUR {int(scores['cost']):,}\ntone: {scores['tone']}\nwriting: {scores['writing']}\n"
             f"language of the email: {scores['language']}")
    for attempt in range(3):     # ask again if the answer is incomplete: COVERED, but no payout
        lines = read_lines(policy_agent(facts))
        decision = lines.get("decision", "?").upper().strip(" .")
        covered = decision.startswith("COVERED")
        payout = euros(lines.get("payout", "")) if covered else 0
        if not (covered and payout == 0):
            break
    return {"facts": facts, "decision": decision, "reason": lines.get("reason", ""), "payout": payout, "covered": covered}

results = {}   # the list of results: each mix of scores is sent to agent 2 only once

def desk_result(scores):
    """The result for six scores: from the list if it is there, otherwise from agent 2."""
    key = tuple(str(scores[name]) for name in NAMES)
    if key not in results:
        results[key] = decide(dict(zip(NAMES, key)))
    return results[key]

print("Claim desk ready: the scoring agent, and agent 2 (policy, with its tool).")

## Step 3 · Upload the emails; the scoring agent scores each one

Click ▶. A button *Choose files* appears: choose **claim_emails.zip** ([download it here](https://github.com/jorisdemmers/multi-agent-claim-desk/raw/main/claim_emails.zip)). The scoring agent reads every email and gives it six scores.

**Look:** do you agree with the scores? (Open a few emails to check.)

In [ ]:
# @title Step 3 · Upload the emails; the scoring agent scores each one
import os, io, zipfile
import pandas as pd
from concurrent.futures import ThreadPoolExecutor

def read_emails(uploaded):
    """From the uploaded files (a zip file, or .txt files) to {file name: email text}."""
    emails = {}
    for name, data in uploaded.items():
        if name.lower().endswith(".zip"):
            with zipfile.ZipFile(io.BytesIO(data)) as box:
                for inner in sorted(box.namelist()):
                    if inner.lower().endswith(".txt") and "__MACOSX" not in inner:
                        emails[os.path.basename(inner)] = box.read(inner).decode("utf-8", errors="ignore")
        else:
            emails[name] = data.decode("utf-8", errors="ignore")
    return emails

try:
    from google.colab import files
    EMAILS = read_emails(files.upload())            # a button appears: choose claim_emails.zip
except ImportError:                                 # outside Colab (a test run): the ten example emails
    folder = os.path.join("data", "claim_emails")
    EMAILS = read_emails({f: open(os.path.join(folder, f), "rb").read() for f in sorted(os.listdir(folder))})
if not EMAILS:
    raise SystemExit("No emails found. Run this step again and choose claim_emails.zip.")

with ThreadPoolExecutor(max_workers=8) as pool:     # 8 emails are scored at the same time
    SCORES = dict(zip(EMAILS, pool.map(score_email, EMAILS.values())))
print(f"The scoring agent's six scores for {len(SCORES)} emails:")
display(pd.DataFrame.from_dict(SCORES, orient="index"))
print(f"So far: {len(calls)} calls, ${total_cost():.4f}.")

## Step 4 · SHAP on all emails

SHAP compares every email with **one reference claim**: hail, roof, EUR 2,500, polite, neat, in Dutch. For each email it splits the difference in payout over the six scores. Agent 2 is asked about every mix it needs.

Click ▶. It takes a minute or two. You will see:
- a **table**: per email, what each score added (+) or took away (−), in euros;
- a **summary per score**: in how many emails it differs from the reference, and its average effect;
- a **chart**: the average size of each score's effect, over all emails.

**Look:** do tone, writing or language take money away? In how many emails?

In [ ]:
# @title Step 4 · SHAP on all emails, against one reference claim
import numpy as np
import shap

REFERENCE = {"cause": "hail", "damage": "roof", "cost": "2500", "tone": "polite", "writing": "neat", "language": "Dutch"}

def as_codes(claim):
    """SHAP works with numbers: each value becomes its place in the list of possible values."""
    return np.array([INPUTS[score].index(str(claim[score])) for score in NAMES])

def payouts(rows):
    """For SHAP: rows of codes in, payouts out. 8 claims go to agent 2 at the same time."""
    claims = [{score: INPUTS[score][int(code)] for score, code in zip(NAMES, row)} for row in rows]
    with ThreadPoolExecutor(max_workers=8) as pool:
        return np.array([result["payout"] for result in pool.map(desk_result, claims)], dtype=float)

calls_before = len(calls)
explainer = shap.explainers.Exact(payouts, shap.maskers.Independent(as_codes(REFERENCE).reshape(1, -1), max_samples=1))
explanation = explainer(np.vstack([as_codes(SCORES[email]) for email in SCORES]))
explanation.feature_names = NAMES
explanation.display_data = np.array([[str(SCORES[email][score]) for score in NAMES] for email in SCORES], dtype=object)

TABLE = pd.DataFrame(explanation.values.round(0).astype(int), index=list(SCORES), columns=NAMES)
TABLE["payout"] = [desk_result(SCORES[email])["payout"] for email in SCORES]
print(f"Reference claim: {REFERENCE}  ->  payout EUR {desk_result(REFERENCE)['payout']:,}")
print("\nWhat each score added to (+) or took away from (-) the payout, in euros, per email:")
display(TABLE)

SUMMARY = []
for j, score in enumerate(NAMES):
    differ = [i for i, email in enumerate(SCORES) if str(SCORES[email][score]) != str(REFERENCE[score])]
    if differ:
        average = explanation.values[differ, j].mean()
        SUMMARY.append(f"{score}: differs from the reference in {len(differ)} of {len(SCORES)} emails; "
                       f"average effect there: EUR {average:+,.0f}.")
    else:
        SUMMARY.append(f"{score}: the same as the reference in all emails.")
print("\nSummary per score:")
for line in SUMMARY:
    print("  " + line)
print(f"\nThis step: {len(calls) - calls_before} calls. Total so far: {len(calls)} calls, ${total_cost():.4f}.")
shap.plots.bar(explanation)    # the average size of each score's effect, over all emails

## Step 5 · Your SHAP report

Click ▶. The notebook writes a one-page PDF with the table, the summary and the chart. Your browser downloads it.

In [ ]:
# @title Step 5 · Your SHAP report (a PDF)
import io, textwrap, datetime
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

table_lines = ["email".ljust(26) + "".join(score[:8].rjust(9) for score in NAMES) + "payout".rjust(9)]
for email, row in TABLE.iterrows():
    table_lines.append(email[:25].ljust(26) + "".join(f"{row[score]:+,}".rjust(9) for score in NAMES) + f"{row['payout']:,}".rjust(9))

text = [("SHAP audit: what changes the payout?", 16, "bold", False),
        (f"Claim desk test, week 5 tutorial. {datetime.date.today():%d %B %Y}. Model: {MODEL}. {len(SCORES)} emails.", 9, "normal", False), ("", 6, "normal", False),
        ("Reference claim: " + ", ".join(f"{score}: {REFERENCE[score]}" for score in NAMES) + f"  ->  EUR {desk_result(REFERENCE)['payout']:,}", 9, "normal", False),
        ("", 6, "normal", False),
        ("What each score added to (+) or took away from (-) the payout, in euros", 11, "bold", False)] + [
        (line, 7.5, "normal", True) for line in table_lines] + [
        ("", 6, "normal", False), ("Summary per score", 11, "bold", False)] + [
        ("    " + line, 9, "normal", False) for line in SUMMARY]

chart = io.BytesIO()                               # the bar chart from Step 4, as a picture
shap.plots.bar(explanation, show=False)
plt.gcf().savefig(chart, format="png", dpi=150, bbox_inches="tight")
plt.close("all")
chart.seek(0)

FILE = "shap_audit_report.pdf"
with PdfPages(FILE) as pdf:
    page = plt.figure(figsize=(8.27, 11.69))       # A4
    y = 0.95
    for line, size, weight, mono in text:
        page.text(0.06, y, line, fontsize=size, fontweight=weight, va="top", family="monospace" if mono else "sans-serif")
        y -= size / 11.69 / 72 * 1.45
    picture = page.add_axes([0.06, 0.04, 0.88, max(0.2, y - 0.07)])
    picture.imshow(plt.imread(chart))
    picture.axis("off")
    pdf.savefig(page)
    plt.close(page)

print(f"Report written: {FILE}")
try:
    from google.colab import files
    files.download(FILE)                           # in Colab: your browser downloads the PDF
except ImportError:
    pass

## Step 6 · Compare and discuss

Put your report next to another pair's.

1. Which scores took money away? In how many emails, and how much?
2. Which emails were not affected by tone, writing or language? Why?
3. SHAP only sees the six scores. Could it find an effect of the customer's name, as in the demo?
4. Would you let agent 2 decide without a person?

**Done early?** Write your own claim email in a .txt file and upload it in Step 3 (with or without the zip). Then run Steps 3 to 5 again.

In [ ]:
# @title The cost of this notebook so far
print(f"Calls to the model: {len(calls)}")
print(f"Cost: ${total_cost():.4f} (budget: ${MAX_COST_TOTAL_USD})")
print(f"Different claims sent to the claim desk: {len(results)}")